# Modelo baseline — TF-IDF + Regressão Logística (multirrótulo)

Este notebook treina o **modelo simples** (a "régua") que classifica cada proposição em um
ou mais dos 32 temas, a partir da **ementa**. Depois, o BERTimbau (próxima fase) tenta superar.

- **Entrada:** `proposicoes_temas.csv` (gerado no notebook de coleta).
- **Saídas:** `resultados_baseline.csv` (métricas) e `particao_treino_val_teste.csv`
  (a divisão treino/validação/teste, reusada pela Fase 4).

**Como rodar:** `pip install scikit-learn pandas iterative-stratification` e rode de cima
para baixo (~1–2 min, sem GPU).

## Passo 1 — Carregar os dados
A coluna `temas` tem os temas separados por `|`; viramos numa lista.

In [9]:
import pandas as pd

dados = pd.read_csv("proposicoes_temas.csv")
dados["ementa"] = dados["ementa"].fillna("").astype(str)
dados["lista_temas"] = dados["temas"].str.split("|")

print(f"{len(dados)} proposicoes carregadas.")
dados[["ementa", "temas"]].head()

19354 proposicoes carregadas.


,ementa,temas
0,Estabelece normas gerais em contratos de segur...,Direito Civil e Processual Civil|Economia
1,Torna obrigatória a homologação em cartório de...,Economia|Previdência e Assistência Social
2,Dispõe sobre a prioridade epidemiológica no tr...,Saúde
3,"Institui o dia 25 de julho como o ""Dia Naciona...",Homenagens e Datas Comemorativas
4,"Altera o art. 121 do Decreto-Lei nº 2.848, de ...",Direito Penal e Processual Penal


## Passo 2 — Rótulos viram tabela de 0 e 1 (multi-hot)
Uma coluna por tema (32). `Y` é o que o modelo aprende a prever.

In [10]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(dados["lista_temas"])
nomes_temas = list(mlb.classes_)
mask_sup200 = Y.sum(axis=0) >= 200   # temas com suporte suficiente p/ avaliar com justica
print(f"Matriz de rotulos: {Y.shape[0]} x {Y.shape[1]} temas")

Matriz de rotulos: 19354 x 32 temas


## Passo 3 — Separar treino, validação e teste (estratificado multirrótulo)

- **Treino (70%)**: aprende. **Validação (15%)**: ajusta escolhas (aqui, os limiares por tema).
- **Teste (15%)**: a prova final, medida uma vez.

Usamos **estratificação iterativa** (`MultilabelStratifiedShuffleSplit`): ela garante que
cada tema apareça **proporcionalmente** nas três partições — importante porque é multirrótulo
e há temas raros. Salvamos a divisão para a Fase 4 usar exatamente o mesmo teste.

In [ ]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
import numpy as np

# 1) separa 15% para TESTE (estratificado pelos 32 temas)
m1 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
idx_resto, idx_teste = next(m1.split(np.zeros(len(Y)), Y))
# 2) do restante, separa ~15% do total para VALIDACAO (0.1765 do resto)
m2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
rel_tr, rel_val = next(m2.split(np.zeros(len(idx_resto)), Y[idx_resto]))
idx_treino, idx_val = idx_resto[rel_tr], idx_resto[rel_val]
print(f"treino: {len(idx_treino)} | validacao: {len(idx_val)} | teste: {len(idx_teste)}")

dados["particao"] = "treino"
dados.loc[idx_val, "particao"] = "validacao"
dados.loc[idx_teste, "particao"] = "teste"
dados[["id", "particao"]].to_csv("particao_treino_val_teste.csv", index=False, encoding="utf-8")
print("Divisao (estratificada) salva em particao_treino_val_teste.csv")

## Passo 4 — Texto em números (TF-IDF)
Aprende o vocabulário **só no treino** e aplica em validação e teste.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

textos = dados["ementa"].values
vectorizer = TfidfVectorizer(min_df=5, ngram_range=(1, 2))
X_treino = vectorizer.fit_transform(textos[idx_treino])
X_val    = vectorizer.transform(textos[idx_val])
X_teste  = vectorizer.transform(textos[idx_teste])

Y_treino, Y_val, Y_teste = Y[idx_treino], Y[idx_val], Y[idx_teste]
print(f"Vocabulario: {len(vectorizer.vocabulary_)} termos")

## Passo 5 — Treinar (Regressão Logística, um-contra-todos)
Um classificador Sim/Não por tema. `class_weight="balanced"` ajuda os temas menos frequentes.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

modelo = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight="balanced"))
modelo.fit(X_treino, Y_treino)
print("Modelo treinado!")

## Passo 6 — Ajustar o limiar de cada tema (na validação)

Por padrão diríamos "tema presente" quando a probabilidade passa de **0,50**, igual para
todos. Mas alguns temas vão melhor com um limiar mais baixo (modelo "tímido") ou mais alto.
Então, **para cada tema**, testamos vários limiares **na validação** e guardamos o que dá o
melhor F1 daquele tema. Depois aplicamos no teste. (Escolher na validação evita "cola".)

In [ ]:
from sklearn.metrics import f1_score

def melhores_limiares(y_val, prob_val, grade=np.arange(0.05, 0.96, 0.05)):
    """Para cada tema, escolhe o limiar que maximiza o F1 na validacao."""
    ts = []
    for j in range(y_val.shape[1]):
        f1s = [f1_score(y_val[:, j], (prob_val[:, j] >= t).astype(int), zero_division=0)
               for t in grade]
        ts.append(grade[int(np.argmax(f1s))] if max(f1s) > 0 else 0.5)
    return np.array(ts)

prob_val   = modelo.predict_proba(X_val)
prob_teste = modelo.predict_proba(X_teste)
limiares = melhores_limiares(Y_val, prob_val)

print("Exemplos de limiar escolhido por tema:")
for nome, t in sorted(zip(nomes_temas, limiares), key=lambda x: x[1])[:6]:
    print(f"  {nome:<40} limiar {t:.2f}")

## Passo 7 — Avaliar no teste: 0,50 vs limiar ajustado
Comparamos o desempenho com o limiar fixo de 0,50 e com os limiares ajustados por tema.

In [ ]:
def resumo(y_true, y_pred):
    return (f1_score(y_true, y_pred, average="macro", zero_division=0),
            f1_score(y_true[:, mask_sup200], y_pred[:, mask_sup200], average="macro", zero_division=0),
            f1_score(y_true, y_pred, average="micro", zero_division=0))

pred_05 = (prob_teste >= 0.5).astype(int)
pred_aj = (prob_teste >= limiares).astype(int)

m05 = resumo(Y_teste, pred_05)
maj = resumo(Y_teste, pred_aj)
print("                         macro(32)  macro(>=200)  micro")
print(f"limiar 0,50:             {m05[0]:.3f}      {m05[1]:.3f}       {m05[2]:.3f}")
print(f"limiar ajustado/tema:    {maj[0]:.3f}      {maj[1]:.3f}       {maj[2]:.3f}")

## Passo 8 — Interpretar: palavras de maior peso por tema
Se as palavras fazem sentido, é sinal de que o modelo aprendeu padrões plausíveis.

In [ ]:
vocab = np.array(vectorizer.get_feature_names_out())
for i, tema in enumerate(nomes_temas):
    est = modelo.estimators_[i]
    if not hasattr(est, "coef_"):
        print(f"{tema}: (poucos exemplos)")
        continue
    top = np.argsort(est.coef_[0])[-8:][::-1]
    print(f"{tema}: {', '.join(vocab[top])}")

## Passo 9 — Salvar os resultados
Salvamos o F1 por tema **no limiar ajustado** (mais o limiar e o F1 a 0,50) em
`resultados_baseline.csv`, para comparar com o BERTimbau.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p_aj, r_aj, f_aj, sup = precision_recall_fscore_support(Y_teste, pred_aj, zero_division=0)
f_05 = precision_recall_fscore_support(Y_teste, pred_05, zero_division=0)[2]

resultados = pd.DataFrame({
    "tema": nomes_temas, "limiar": limiares,
    "precisao": p_aj, "revocacao": r_aj, "f1": f_aj, "f1_lim050": f_05,
    "suporte_teste": sup,
}).sort_values("f1", ascending=False)

resultados.to_csv("resultados_baseline.csv", index=False, encoding="utf-8")
print("Salvo: resultados_baseline.csv")
resultados